# Use Case 4 — Critical Path & Bottleneck Analysis

**Goal:** Identify the critical path in the EPC project using PRECEDES relations, and find scheduling bottlenecks by discipline and permit type  
**Neo4j:** `bolt://172.22.43.151:7687`  
**Prerequisite:** `import_graph_real.py` must have been run

## Questions answered:
1. What is the **critical path** (longest PRECEDES chain)?
2. Which **disciplines** concentrate the most constrained steps?
3. Which **permit types** create the most downstream dependencies (bottlenecks)?
4. Which **activities** lie on the critical path?

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from neo4j import GraphDatabase
import os
import warnings
warnings.filterwarnings('ignore')

NEO4J_URI      = 'bolt://172.22.43.151:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'your_password'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(cypher, **params):
    with driver.session() as s:
        result = s.run(cypher, **params)
        return pd.DataFrame([r.data() for r in result])

os.makedirs('../../experiments/UseCase4', exist_ok=True)
print('✅ Connected to Neo4j')

## 1. Q5 — Critical Path (Longest PRECEDES Chain)

In [ ]:
cp = run_query('''
    MATCH path = (start:Step)-[:PRECEDES*..30]->(end:Step)
    WHERE NOT ()-[:PRECEDES]->(start)
      AND NOT (end)-[:PRECEDES]->()
    WITH path, length(path) AS depth
    ORDER BY depth DESC
    LIMIT 1
    RETURN [n IN nodes(path) | n.name]        AS step_names,
           [n IN nodes(path) | n.activity_id] AS activity_ids,
           [n IN nodes(path) | n.permit_type] AS permit_types,
           [n IN nodes(path) | n.id]          AS step_ids,
           depth
''')

if len(cp) > 0:
    depth       = cp['depth'].iloc[0]
    step_names  = cp['step_names'].iloc[0]
    activity_ids= cp['activity_ids'].iloc[0]
    permit_types= cp['permit_types'].iloc[0]
    print(f'Critical path length: {depth} steps')
    print()
    print(f'{"Step":<4} {"Activity":<12} {"Permit":<18} {"Name"}')
    print('-'*70)
    for i, (n, a, p) in enumerate(zip(step_names, activity_ids, permit_types)):
        print(f'{i+1:<4} {a:<12} {p:<18} {n}')
else:
    print('⚠️  No PRECEDES chains found — check import')

In [ ]:
PERMIT_COLORS = {
    'hot_work':       '#f38ba8',
    'excavation':     '#fab387',
    'lifting':        '#f9e2af',
    'electrical':     '#a6e3a1',
    'confined_space': '#74c7ec',
    'radiography':    '#cba6f7',
    'work_at_height': '#89dceb',
    'general_work':   '#6c7086',
}

if len(cp) > 0:
    n_steps = len(step_names)
    fig, ax = plt.subplots(figsize=(max(14, n_steps * 1.5), 4))
    fig.patch.set_facecolor('#1e1e2e')
    ax.set_facecolor('#181825')
    ax.axis('off')

    show_steps = step_names[:12]  # limit display for readability
    show_acts  = activity_ids[:12]
    show_perms = permit_types[:12]

    for i, (name, act, perm) in enumerate(zip(show_steps, show_acts, show_perms)):
        color = PERMIT_COLORS.get(perm, '#cdd6f4')
        x = i * 2.8
        ax.add_patch(mpatches.FancyBboxPatch(
            (x, 0.3), 2.5, 0.4,
            boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='#313244'
        ))
        short_name = name[:18] + '…' if len(name) > 18 else name
        ax.text(x+1.25, 0.52, short_name,
                ha='center', va='center', color='#1e1e2e', fontsize=6.5, fontweight='bold')
        ax.text(x+1.25, 0.37, act,
                ha='center', va='center', color='#1e1e2e', fontsize=6)
        if i < len(show_steps)-1:
            ax.annotate('', xy=(x+2.8, 0.5), xytext=(x+2.5, 0.5),
                        arrowprops=dict(arrowstyle='->', color='#cdd6f4', lw=1.5))

    if n_steps > 12:
        ax.text(len(show_steps)*2.8 + 0.2, 0.5, f'… +{n_steps-12} more',
                va='center', color='#cdd6f4', fontsize=9)

    legend_patches = [mpatches.Patch(color=c, label=p.replace('_',' '))
                      for p, c in PERMIT_COLORS.items()]
    ax.legend(handles=legend_patches, loc='lower center', ncol=4,
              facecolor='#313244', labelcolor='#cdd6f4', fontsize=7,
              bbox_to_anchor=(0.5, -0.15))

    ax.set_xlim(-0.3, len(show_steps)*2.8 + 1)
    ax.set_ylim(0.1, 0.9)
    ax.set_title(f'Critical Path — {depth} steps (showing first {min(12,n_steps)})',
                 color='#cdd6f4', fontsize=11)
    plt.tight_layout()
    plt.savefig('../../experiments/UseCase4/critical_path.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Q6 — Bottleneck by Discipline

In [ ]:
disc_bottleneck = run_query('''
    MATCH (act:Activity)-[:HAS_STEP]->(s:Step)-[:REQUIRES_PERMIT]->(wp:WorkPermit)
    RETURN act.discipline AS discipline,
           wp.name        AS permit,
           count(s)       AS n_steps
    ORDER BY discipline, n_steps DESC
''')

print('Steps per discipline per permit type:')
if disc_bottleneck.empty or 'n_steps' not in disc_bottleneck.columns:
    print('  (no steps with permits found — check REQUIRES_PERMIT relationships)')
else:
    print(disc_bottleneck.pivot_table(
        index='discipline', columns='permit', values='n_steps', fill_value=0
    ).to_string())

In [ ]:
if disc_bottleneck.empty or 'n_steps' not in disc_bottleneck.columns:
    print('Skipping chart — no discipline/permit data')
else:
    pivot = disc_bottleneck.pivot_table(
        index='discipline', columns='permit', values='n_steps', fill_value=0
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#1e1e2e')
    ax.set_facecolor('#181825')

    bottom = np.zeros(len(pivot))
    for permit in pivot.columns:
        color = PERMIT_COLORS.get(permit, '#cdd6f4')
        ax.bar(pivot.index, pivot[permit], bottom=bottom,
               color=color, edgecolor='#313244', label=permit.replace('_',' '))
        bottom += pivot[permit].values

    ax.set_title('Steps by Discipline and Permit Type', color='#cdd6f4', fontsize=12)
    ax.set_xlabel('Discipline', color='#cdd6f4')
    ax.set_ylabel('Number of Steps', color='#cdd6f4')
    ax.tick_params(colors='#cdd6f4')
    ax.legend(loc='upper right', facecolor='#313244', labelcolor='#cdd6f4', fontsize=8)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    for spine in ['bottom','left']: ax.spines[spine].set_color('#313244')

    plt.tight_layout()
    plt.savefig('../../experiments/UseCase4/bottleneck_discipline.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Q7 — Most Blocking Steps (Highest Downstream Count)

In [ ]:
blocking = run_query('''
    MATCH (s:Step)-[:PRECEDES*1..30]->(downstream:Step)
    WITH s, count(downstream) AS blocks
    ORDER BY blocks DESC
    LIMIT 10
    MATCH (s)-[:REQUIRES_PERMIT]->(wp:WorkPermit)
    RETURN s.id AS step_id, s.name AS step_name,
           s.activity_id AS activity, wp.id AS permit, blocks
    ORDER BY blocks DESC
''')

print('Top 10 most blocking steps:')
print(blocking.to_string(index=False))

In [ ]:
if blocking.empty:
    print('⚠️  Q7 (original): no blocking steps found — inner MATCH dropped steps without permits')
    print('    See Q7 Corrected below (Section 5) which uses OPTIONAL MATCH')
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#1e1e2e')
    ax.set_facecolor('#181825')

    colors = [PERMIT_COLORS.get(p, '#cdd6f4') for p in blocking['permit']]
    labels = [f"{row['activity']}\n{row['step_name'][:20]}" for _, row in blocking.iterrows()]

    bars = ax.barh(labels, blocking['blocks'], color=colors, edgecolor='#313244')
    for bar, val in zip(bars, blocking['blocks']):
        ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                str(val), va='center', color='#cdd6f4', fontsize=9)

    ax.set_xlabel('Downstream steps blocked', color='#cdd6f4')
    ax.set_title('Top 10 Most Blocking Steps\n(color = permit type required)',
                 color='#cdd6f4', fontsize=11)
    ax.tick_params(colors='#cdd6f4', labelsize=8)
    ax.invert_yaxis()
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    for spine in ['bottom','left']: ax.spines[spine].set_color('#313244')

    legend_patches = [mpatches.Patch(color=c, label=p.replace('_',' '))
                      for p, c in PERMIT_COLORS.items()
                      if p in blocking['permit'].values]
    ax.legend(handles=legend_patches, facecolor='#313244', labelcolor='#cdd6f4', fontsize=8)

    plt.tight_layout()
    plt.savefig('../../experiments/UseCase4/bottleneck_steps.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Q8 — Activities on Critical Path

In [ ]:
cp_activities = run_query('''
    MATCH path = (start:Step)-[:PRECEDES*..30]->(end:Step)
    WHERE NOT ()-[:PRECEDES]->(start)
      AND NOT (end)-[:PRECEDES]->()
    WITH path, length(path) AS depth
    ORDER BY depth DESC
    LIMIT 5
    UNWIND nodes(path) AS s
    MATCH (act:Activity)-[:HAS_STEP]->(s)
    RETURN DISTINCT act.id AS activity_id, act.name AS activity_name,
                    act.discipline AS discipline, depth
    ORDER BY depth DESC, discipline
''')

print('Activities on top-5 critical paths:')
print(cp_activities.to_string(index=False))

## 5. Summary

| Analysis | Key Finding |
|---|---|
| Critical path length | see Q5 output |
| Most constrained discipline | see Q6 heatmap |
| Most blocking permit type | see Q7 chart |
| Activities on critical path | see Q8 output |

**Tesi key point:** the PRECEDES graph enables scheduling analysis that goes beyond what a flat activity list can provide — the TKG structure directly encodes sequencing constraints and allows graph algorithms (longest path, centrality) to identify project risk.

## 5. Corrections — Duration-Based Critical Path

The original Q5 and Q8 used **hop count** (number of steps) as the critical path metric. This is incorrect: in project management, the critical path is the sequence with **maximum total duration** (sum of `estimated_hours`). Q7 used an inner MATCH for permits, silently dropping the most blocking steps if they had no permit assigned.

| Query | Original | Corrected |
|-------|----------|-----------|
| Q5 | `length(path)` longest path | `reduce(total_hours)` max duration path |
| Q7 | `MATCH` permit (inner join) | `OPTIONAL MATCH` permit (keeps all steps) |
| Q8 | `ORDER BY depth DESC` | `ORDER BY total_hours DESC` |

In [ ]:

# Q5 Corrected — Critical path by total estimated_hours (not hop count)
cp_corr = run_query('''
    MATCH path = (start:Step)-[:PRECEDES*..30]->(end:Step)
    WHERE NOT ()-[:PRECEDES]->(start)
      AND NOT (end)-[:PRECEDES]->()
    WITH path,
         reduce(total=0, n IN nodes(path) | total + coalesce(n.estimated_hours, 0)) AS total_hours,
         length(path) AS depth
    ORDER BY total_hours DESC
    LIMIT 1
    RETURN [n IN nodes(path) | n.name]        AS step_names,
           [n IN nodes(path) | n.activity_id] AS activity_ids,
           [n IN nodes(path) | n.permit_type] AS permit_types,
           [n IN nodes(path) | n.id]          AS step_ids,
           total_hours,
           depth
''')

if len(cp_corr) > 0:
    total_h   = cp_corr['total_hours'].iloc[0]
    depth_c   = cp_corr['depth'].iloc[0]
    step_names_c  = cp_corr['step_names'].iloc[0]
    activity_ids_c= cp_corr['activity_ids'].iloc[0]
    permit_types_c= cp_corr['permit_types'].iloc[0]
    print(f'Critical path (by duration): {total_h:.0f} estimated hours, {depth_c} steps')
    print()
    print(f'{"#":<4} {"Activity":<12} {"Permit":<18} {"Name"}')
    print('-'*70)
    for i, (n, a, p) in enumerate(zip(step_names_c, activity_ids_c, permit_types_c)):
        print(f'{i+1:<4} {a:<12} {str(p):<18} {n}')

    n_steps_c = len(step_names_c)
    fig, ax = plt.subplots(figsize=(max(14, n_steps_c * 1.5), 4))
    fig.patch.set_facecolor('#1e1e2e')
    ax.set_facecolor('#181825')
    ax.axis('off')
    show_steps_c  = step_names_c[:12]
    show_acts_c   = activity_ids_c[:12]
    show_perms_c  = permit_types_c[:12]
    for i, (name, act, perm) in enumerate(zip(show_steps_c, show_acts_c, show_perms_c)):
        color = PERMIT_COLORS.get(perm, '#cdd6f4')
        x = i * 2.8
        ax.add_patch(mpatches.FancyBboxPatch(
            (x, 0.3), 2.5, 0.4, boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='#313244'))
        short_name = name[:18] + '…' if len(name) > 18 else name
        ax.text(x+1.25, 0.52, short_name,
                ha='center', va='center', color='#1e1e2e', fontsize=6.5, fontweight='bold')
        ax.text(x+1.25, 0.37, str(act),
                ha='center', va='center', color='#1e1e2e', fontsize=6)
        if i < len(show_steps_c)-1:
            ax.annotate('', xy=(x+2.8, 0.5), xytext=(x+2.5, 0.5),
                        arrowprops=dict(arrowstyle='->', color='#cdd6f4', lw=1.5))
    if n_steps_c > 12:
        ax.text(len(show_steps_c)*2.8 + 0.2, 0.5, f'… +{n_steps_c-12} more',
                va='center', color='#cdd6f4', fontsize=9)
    legend_patches = [mpatches.Patch(color=c, label=p.replace('_',' '))
                      for p, c in PERMIT_COLORS.items()]
    ax.legend(handles=legend_patches, loc='lower center', ncol=4,
              facecolor='#313244', labelcolor='#cdd6f4', fontsize=7,
              bbox_to_anchor=(0.5, -0.15))
    ax.set_xlim(-0.3, len(show_steps_c)*2.8 + 1)
    ax.set_ylim(0.1, 0.9)
    ax.set_title(
        f'Q5 Corrected — Critical Path by Duration: {total_h:.0f}h total, {depth_c} steps'
        f' (showing first {min(12,n_steps_c)})',
        color='#cdd6f4', fontsize=11)
    plt.tight_layout()
    plt.savefig('../../experiments/UseCase4/critical_path_corrected.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  No PRECEDES chains found')


In [ ]:

# Q7 Corrected — Most blocking steps with OPTIONAL MATCH (includes steps with no permit)
blocking_corr = run_query('''
    MATCH (s:Step)-[:PRECEDES*1..30]->(downstream:Step)
    WITH s, count(downstream) AS blocks
    ORDER BY blocks DESC
    LIMIT 10
    OPTIONAL MATCH (s)-[:REQUIRES_PERMIT]->(wp:WorkPermit)
    RETURN s.id AS step_id, s.name AS step_name,
           s.activity_id AS activity,
           coalesce(wp.id, 'none') AS permit,
           blocks
    ORDER BY blocks DESC
''')

print('Top 10 most blocking steps (OPTIONAL permit join):')
print(blocking_corr.to_string(index=False))

if len(blocking_corr) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#1e1e2e')
    ax.set_facecolor('#181825')
    colors = [PERMIT_COLORS.get(p, '#cdd6f4') for p in blocking_corr['permit']]
    labels = [f"{row['activity']}\n{row['step_name'][:20]}" for _, row in blocking_corr.iterrows()]
    bars = ax.barh(labels, blocking_corr['blocks'], color=colors, edgecolor='#313244')
    for bar, val in zip(bars, blocking_corr['blocks']):
        ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                str(val), va='center', color='#cdd6f4', fontsize=9)
    ax.set_xlabel('Downstream steps blocked', color='#cdd6f4')
    ax.set_title('Q7 Corrected — Top 10 Most Blocking Steps\n(OPTIONAL MATCH: includes steps without permit)',
                 color='#cdd6f4', fontsize=11)
    ax.tick_params(colors='#cdd6f4', labelsize=8)
    ax.invert_yaxis()
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    for spine in ['bottom','left']: ax.spines[spine].set_color('#313244')
    legend_patches = [mpatches.Patch(color=c, label=p.replace('_',' '))
                      for p, c in PERMIT_COLORS.items()
                      if p in blocking_corr['permit'].values]
    if legend_patches:
        ax.legend(handles=legend_patches, facecolor='#313244', labelcolor='#cdd6f4', fontsize=8)
    plt.tight_layout()
    plt.savefig('../../experiments/UseCase4/bottleneck_steps_corrected.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:

# Q8 Corrected — Activities on duration-based critical path (top-5 paths by total_hours)
cp_activities_corr = run_query('''
    MATCH path = (start:Step)-[:PRECEDES*..30]->(end:Step)
    WHERE NOT ()-[:PRECEDES]->(start)
      AND NOT (end)-[:PRECEDES]->()
    WITH path,
         reduce(total=0, n IN nodes(path) | total + coalesce(n.estimated_hours, 0)) AS total_hours,
         length(path) AS depth
    ORDER BY total_hours DESC
    LIMIT 5
    UNWIND nodes(path) AS s
    MATCH (act:Activity)-[:HAS_STEP]->(s)
    RETURN DISTINCT act.id AS activity_id, act.name AS activity_name,
                    act.discipline AS discipline, total_hours, depth
    ORDER BY total_hours DESC, discipline
''')

print('Activities on top-5 duration-critical paths:')
print(cp_activities_corr.to_string(index=False))


In [ ]:
driver.close()
print('✅ Done — results saved to experiments/UseCase4/')